<div style="background:linear-gradient(135deg,#ffffff,#fffdfd);color:#5c1d0c;padding:22px;border-left:6px solid #E24307;border-radius:12px;font-family:'Segoe UI',sans-serif;box-shadow:0px 4px 12px rgba(0,0,0,0.15);">
  <div style="display:flex;align-items:center;gap:12px;">
    <img src="data/logo-swift.png" alt="Logo Swift" style="height:45px;">
    <h2 style="color:#E24307;margin:0;letter-spacing:1px;font-size:24px;">Modelo de Análise de Sentimento</h2>
  </div>
  <hr style="border:0;border-top:1px solid #f4b39a;margin:15px 0;">
  <p style="margin:8px 0;"><b style="color:#a12d00;">Membros:</b>
    <span style="color:#5f3a2d;font-size:13px;">Bruna Carvalho Cardoso | Giovanne Antony Bahia Torquato | Kaique Moreira Arantes De Souza | Letícia Nascimento da Silva | Sofia Bonini Pinto</span>
  </p>
  <p style="margin:8px 0;"><b style="color:#a12d00;">Data Entrega:</b>
    <span style="color:#5f3a2d;font-size:13px;">09/06/2026</span>
  </p>
</div>

<div style="background-color:#fff4ee;padding:20px;border-radius:12px;font-family:Arial,sans-serif;border-left:6px solid #E24307;box-shadow:0 2px 8px rgba(0,0,0,0.08);">
  <h2 style="color:#E24307;margin:0;font-size:24px;">0. Importações e Configurações</h2>
</div>

In [2]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_class_weight

# Paleta Swift
PALETTE = {'promotor': '#E24307', 'neutro': '#f39c12', 'detrator': '#5c1d0c'}
LABEL_NAMES = ['Negativo', 'Neutro', 'Positivo']  # 0, 1, 2
LABEL_CORES = ['#5c1d0c', '#f39c12', '#E24307']

RANDOM_STATE = 42

print('✅ Ambiente configurado.')

✅ Ambiente configurado.


<div style="background-color:#fff4ee;padding:20px;border-radius:12px;font-family:Arial,sans-serif;border-left:6px solid #E24307;box-shadow:0 2px 8px rgba(0,0,0,0.08);">
  <h2 style="color:#E24307;margin:0;font-size:24px;">1. Carregamento e Preparação dos Dados</h2>
</div>

In [3]:
CSV_PATH = 'base_final copy.csv'

df_raw = pd.read_csv(CSV_PATH, sep=';')
df_com = df_raw[df_raw['tem_comentario'] == True].copy()

# Remove os poucos nulos em comentario_limpo
df_com = df_com.dropna(subset=['comentario_limpo'])
df_com['comentario_limpo'] = df_com['comentario_limpo'].astype(str)
df_com['comentario']       = df_com['comentario'].astype(str)

# Label numérico: 0=detrator, 1=neutro, 2=promotor
LABEL_MAP = {'detrator': 0, 'neutro': 1, 'promotor': 2}
df_com['label'] = df_com['classificacao'].map(LABEL_MAP)

print(f'Comentários disponíveis: {len(df_com):,}')
print()
print('Distribuição de classes:')
dist = df_com['label'].value_counts().sort_index()
for lbl, cnt in dist.items():
    print(f'  {LABEL_NAMES[lbl]:<10}: {cnt:>8,}  ({cnt/len(df_com)*100:.1f}%)')

FileNotFoundError: [Errno 2] No such file or directory: 'base_final copy.csv'

<div style="background-color:#fff4ee;padding:20px;border-radius:12px;font-family:Arial,sans-serif;border-left:6px solid #E24307;box-shadow:0 2px 8px rgba(0,0,0,0.08);">
  <h2 style="color:#E24307;margin:0;font-size:24px;">2. Divisão Treino / Validação / Teste</h2>
</div>

**Decisão documentada:**
- Split estratificado por classe: **70% treino / 15% validação / 15% teste**
- Estratificado (`stratify=y`) para preservar a proporção de classes em todos os conjuntos
- Nenhum dado de teste é visto durante o desenvolvimento — avaliação final apenas ao final

In [ ]:
X = df_com['comentario_limpo'].values
y = df_com['label'].values

# Primeiro split: 70% treino | 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE
)

# Segundo split: 50% de temp → validação | 50% → teste  (= 15% / 15% do total)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=RANDOM_STATE
)

print(f'Treino     : {len(X_train):>8,} ({len(X_train)/len(X)*100:.0f}%)')
print(f'Validação  : {len(X_val):>8,} ({len(X_val)/len(X)*100:.0f}%)')
print(f'Teste      : {len(X_test):>8,} ({len(X_test)/len(X)*100:.0f}%)')
print()

# Confirma estratificação
print('Distribuição por split:')
for nome, yy in [('Treino', y_train), ('Validação', y_val), ('Teste', y_test)]:
    uq, cts = np.unique(yy, return_counts=True)
    desc = '  |  '.join([f'{LABEL_NAMES[u]}: {c/len(yy)*100:.1f}%' for u, c in zip(uq, cts)])
    print(f'  {nome:<10}: {desc}')

<div style="background-color:#fff4ee;padding:20px;border-radius:12px;font-family:Arial,sans-serif;border-left:6px solid #E24307;box-shadow:0 2px 8px rgba(0,0,0,0.08);">
  <h2 style="color:#E24307;margin:0;font-size:24px;">3. Baseline — TF-IDF + Regressão Logística</h2>
</div>

**Por que um baseline?**
O documento exige a comparação do modelo final com ao menos um baseline simples. O TF-IDF + Regressão Logística é o ponto de partida canônico para classificação de texto: rápido, interpretável e surpreendentemente competitivo em dados com vocabulário rico.

**Decisões documentadas:**
- `class_weight='balanced'` para compensar o desbalanceamento (90.8% promotores)
- TF-IDF com unigramas e bigramas (`ngram_range=(1,2)`), máximo de 50.000 features
- Critério de parada da LR: `max_iter=1000`

In [ ]:
# Pipeline: TF-IDF → Regressão Logística
baseline_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=50_000,
        sublinear_tf=True,
        min_df=2
    )),
    ('clf', LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=RANDOM_STATE,
        solver='lbfgs',
    ))
])

print('Treinando baseline (TF-IDF + LR)...')
baseline_pipeline.fit(X_train, y_train)
print('✅ Baseline treinado.')

In [ ]:
# Avaliação do baseline no conjunto de VALIDAÇÃO
y_pred_baseline_val = baseline_pipeline.predict(X_val)

print('=== Baseline — Resultados na Validação ===')
print()
print(classification_report(
    y_val, y_pred_baseline_val,
    target_names=LABEL_NAMES
))

f1_baseline_val = f1_score(y_val, y_pred_baseline_val, average='macro')
print(f'F1-Macro (validação): {f1_baseline_val:.4f}')

In [ ]:
# Matriz de confusão — baseline validação
cm_baseline = confusion_matrix(y_val, y_pred_baseline_val)

fig, ax = plt.subplots(figsize=(7, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_baseline, display_labels=LABEL_NAMES)
disp.plot(ax=ax, colorbar=False, cmap='Oranges')
ax.set_title('Baseline (TF-IDF + LR) — Matriz de Confusão (Validação)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Top features mais importantes por classe (TF-IDF)
tfidf       = baseline_pipeline.named_steps['tfidf']
clf         = baseline_pipeline.named_steps['clf']
feature_names = np.array(tfidf.get_feature_names_out())

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Top 15 Features por Classe — Baseline TF-IDF', fontweight='bold')

for i, (nome, cor) in enumerate(zip(LABEL_NAMES, LABEL_CORES)):
    coefs   = clf.coef_[i]
    top_idx = np.argsort(coefs)[-15:]
    top_feat = feature_names[top_idx]
    top_vals = coefs[top_idx]

    axes[i].barh(top_feat, top_vals, color=cor, edgecolor='white')
    axes[i].set_title(f'Classe: {nome}', fontweight='bold')
    axes[i].set_xlabel('Coeficiente')

plt.tight_layout()
plt.show()

<div style="background-color:#fff4ee;padding:20px;border-radius:12px;font-family:Arial,sans-serif;border-left:6px solid #E24307;box-shadow:0 2px 8px rgba(0,0,0,0.08);">
  <h2 style="color:#E24307;margin:0;font-size:24px;">4. Modelo Final — BERTimbau (Fine-tuning)</h2>
</div>

**Arquitetura escolhida: BERTimbau**

A EDA identificou sobreposição de 40% no vocabulário entre detratores e promotores — palavras como "atendimento", "preço" e "qualidade" aparecem nas duas classes com sentidos opostos dependendo do contexto. Modelos baseados em frequência (TF-IDF) não capturam essa ambiguidade. O BERTimbau (`neuralmind/bert-base-portuguese-cased`) lida com isso via atenção bidirecional, lendo cada palavra no contexto completo da frase.

**Decisões de arquitetura documentadas:**
- Modelo base: `neuralmind/bert-base-portuguese-cased`
- Camada de classificação: linear sobre o token `[CLS]` → 3 classes (softmax)
- Entrada: texto original com acentos e pontuação (o BERTimbau foi treinado com texto natural)
- Truncagem: máximo de 128 tokens (cobre >95% dos comentários da base)

**Hiperparâmetros ajustados:**

| Parâmetro | Valor | Justificativa |
|---|---|---|
| Learning rate | `3e-5` | Testado empiricamente — melhor que o padrão `2e-5` para este dataset |
| Epochs | `4` | Critério de parada: menor val_loss entre épocas |
| Batch size | `32` | Padrão para fine-tuning BERT com GPU T4 |
| Max tokens | `128` | Cobre >95% dos comentários; truncagem elimina apenas casos extremos |
| Weight decay | `0.01` | Regularização padrão do AdamW |
| Warmup | `10% dos steps` | Evita instabilidade no início do treino |

**Estratégia de desbalanceamento:**

Pesos manuais na `CrossEntropyLoss`: `[3.0, 3.5, 1.0]` para Negativo, Neutro e Positivo respectivamente. A escolha de pesos manuais (em vez de `class_weight='balanced'` automático) permitiu ajustar o equilíbrio entre recall do Negativo e precisão do Neutro — classes com comportamentos diferentes de erro.

**Critério de parada:**

O modelo com menor `val_loss` entre as épocas é salvo como checkpoint. Ao final do treino, esse checkpoint é carregado para a avaliação no teste — não necessariamente o modelo da última época.

In [ ]:
# Verifica disponibilidade das dependências
try:
    import torch
    from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup
    from torch.utils.data import Dataset, DataLoader
    from torch.optim import AdamW

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'✅ PyTorch {torch.__version__} | Dispositivo: {device}')

    if device.type == 'cpu':
        print()
        print('⚠️  Sem GPU detectada. O fine-tuning do BERTimbau em CPU pode levar horas.')
        print('   Recomendações:')
        print('   - Google Colab (GPU T4 gratuita): https://colab.research.google.com')
        print('   - Kaggle Notebooks (GPU P100 gratuita): https://kaggle.com')
        print('   - Execute este notebook em ambiente com GPU para resultado prático.')
        print()
        print('   O código está completo e pronto para rodar em GPU.')
        print('   Na célula seguinte é possível reduzir SAMPLE_SIZE para testar localmente.')

    BERT_DISPONIVEL = True

except ImportError:
    print('❌ PyTorch ou transformers não instalados.')
    print('   Instale com: pip install torch transformers')
    BERT_DISPONIVEL = False

In [ ]:
if not BERT_DISPONIVEL:
    raise RuntimeError('Instale torch e transformers antes de continuar.')

# ── Configurações ─────────────────────────────────────────────────────────────
MODEL_NAME  = 'neuralmind/bert-base-portuguese-cased'
MAX_LEN     = 128      # cobre > 95% dos comentários
BATCH_SIZE  = 32
EPOCHS      = 4
LR          = 3e-5
NUM_CLASSES = 3

# Para testar localmente sem GPU, reduza SAMPLE_SIZE (ex: 5000)
# Para treino completo, use None
SAMPLE_SIZE = None

# ── Dataset ───────────────────────────────────────────────────────────────────
class NpsDataset(Dataset):
    def __init__(self, textos, labels, tokenizer, max_len):
        self.textos    = textos
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.textos)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.textos[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }

print(f'Configurações BERT: model={MODEL_NAME}, max_len={MAX_LEN}, batch={BATCH_SIZE}, epochs={EPOCHS}, lr={LR}')

In [ ]:
# ── Carrega tokenizer e modelo ────────────────────────────────────────────────
print(f'Carregando tokenizer: {MODEL_NAME}...')
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

# Reduz dataset se SAMPLE_SIZE definido (para testes locais)
if SAMPLE_SIZE:
    idx_tr = np.random.choice(len(X_train), min(SAMPLE_SIZE, len(X_train)), replace=False)
    idx_vl = np.random.choice(len(X_val),   min(SAMPLE_SIZE // 5, len(X_val)),   replace=False)
    X_tr, y_tr = X_train[idx_tr], y_train[idx_tr]
    X_vl, y_vl = X_val[idx_vl],   y_val[idx_vl]
else:
    X_tr, y_tr = X_train, y_train
    X_vl, y_vl = X_val,   y_val

train_dataset = NpsDataset(X_tr.tolist(), y_tr.tolist(), tokenizer, MAX_LEN)
val_dataset   = NpsDataset(X_vl.tolist(), y_vl.tolist(), tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'✅ Tokenizer carregado.')
print(f'   Treino  : {len(train_dataset):,} amostras  |  {len(train_loader):,} batches')
print(f'   Validação: {len(val_dataset):,} amostras  |  {len(val_loader):,} batches')

In [ ]:
# ── Pesos de classe para CrossEntropyLoss ─────────────────────────────────────
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1, 2]),
    y=y_tr
)
weights_tensor = torch.tensor([4.0, 2.0, 1.0], dtype=torch.float).to(device)
print('Pesos de classe:', {LABEL_NAMES[i]: f'{w:.3f}' for i, w in enumerate(class_weights)})

# ── Modelo ────────────────────────────────────────────────────────────────────
print(f'\nCarregando modelo: {MODEL_NAME}...')
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True
).to(device)
print('✅ Modelo carregado e movido para', device)

In [ ]:
# ── Otimizador, scheduler e loss ─────────────────────────────────────────────
optimizer  = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler  = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)
loss_fn = torch.nn.CrossEntropyLoss(weight=weights_tensor)

print(f'Total de steps de treino: {total_steps:,}')
print(f'Warmup steps            : {int(0.1 * total_steps):,}')

In [ ]:
# ── Loop de treinamento ───────────────────────────────────────────────────────
def avaliar(model, loader, loss_fn, device):
    model.eval()
    total_loss, preds_all, labels_all = 0.0, [], []
    with torch.no_grad():
        for batch in loader:
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            lbls = batch['label'].to(device)
            out  = model(input_ids=ids, attention_mask=mask)
            loss = loss_fn(out.logits, lbls)
            total_loss += loss.item()
            preds_all.extend(out.logits.argmax(dim=1).cpu().numpy())
            labels_all.extend(lbls.cpu().numpy())
    avg_loss = total_loss / len(loader)
    f1_mac   = f1_score(labels_all, preds_all, average='macro')
    return avg_loss, f1_mac, np.array(preds_all), np.array(labels_all)


historico = {'epoch': [], 'train_loss': [], 'val_loss': [], 'val_f1_macro': []}
melhor_val_loss = float('inf')
melhor_modelo_path = 'bert_sentimento_melhor.pt'

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0

    for step, batch in enumerate(train_loader, 1):
        ids  = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        lbls = batch['label'].to(device)

        optimizer.zero_grad()
        out  = model(input_ids=ids, attention_mask=mask)
        loss = loss_fn(out.logits, lbls)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        train_loss += loss.item()

        if step % 200 == 0:
            print(f'  Epoch {epoch} | Step {step}/{len(train_loader)} | Loss: {loss.item():.4f}')

    avg_train = train_loss / len(train_loader)
    val_loss, val_f1, _, _ = avaliar(model, val_loader, loss_fn, device)

    historico['epoch'].append(epoch)
    historico['train_loss'].append(avg_train)
    historico['val_loss'].append(val_loss)
    historico['val_f1_macro'].append(val_f1)

    print(f'\nEpoch {epoch}/{EPOCHS} — Train Loss: {avg_train:.4f} | Val Loss: {val_loss:.4f} | Val F1-Macro: {val_f1:.4f}')

    # Critério de parada: salva o modelo com menor val_loss
    if val_loss < melhor_val_loss:
        melhor_val_loss = val_loss
        torch.save(model.state_dict(), melhor_modelo_path)
        print(f'  ✅ Melhor modelo salvo (val_loss={val_loss:.4f})')
    print()

In [ ]:
# Curvas de aprendizado
hist_df = pd.DataFrame(historico)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Curvas de Aprendizado — BERTimbau', fontweight='bold')

axes[0].plot(hist_df['epoch'], hist_df['train_loss'], marker='o', label='Train Loss', color='#E24307')
axes[0].plot(hist_df['epoch'], hist_df['val_loss'],   marker='s', label='Val Loss',   color='#5c1d0c', linestyle='--')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss por Época')
axes[0].legend()
axes[0].set_xticks(hist_df['epoch'])

axes[1].plot(hist_df['epoch'], hist_df['val_f1_macro'], marker='o', color='#f39c12')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('F1-Macro')
axes[1].set_title('F1-Macro (Validação) por Época')
axes[1].set_xticks(hist_df['epoch'])

plt.tight_layout()
plt.show()

<div style="background-color:#fff4ee;padding:20px;border-radius:12px;font-family:Arial,sans-serif;border-left:6px solid #E24307;box-shadow:0 2px 8px rgba(0,0,0,0.08);">
  <h2 style="color:#E24307;margin:0;font-size:24px;">5. Avaliação Final no Conjunto de Teste</h2>
</div>

**Atenção:** esta seção usa o conjunto de teste — dados que nenhum modelo viu durante o desenvolvimento. A avaliação aqui é a definitiva.

In [ ]:
# ── Carrega o melhor checkpoint do BERTimbau ──────────────────────────────────
model.load_state_dict(torch.load(melhor_modelo_path, map_location=device))
model.eval()

test_dataset = NpsDataset(X_test.tolist(), y_test.tolist(), tokenizer, MAX_LEN)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

_, _, y_pred_bert, y_true_test = avaliar(model, test_loader, loss_fn, device)

print('=== BERTimbau — Resultados no TESTE ===')
print()
print(classification_report(y_true_test, y_pred_bert, target_names=LABEL_NAMES))

f1_bert_test = f1_score(y_true_test, y_pred_bert, average='macro')
print(f'F1-Macro (teste): {f1_bert_test:.4f}')

In [ ]:
# ── Baseline no mesmo conjunto de teste (comparação justa) ───────────────────
y_pred_baseline_test = baseline_pipeline.predict(X_test)

print('=== Baseline (TF-IDF + LR) — Resultados no TESTE ===')
print()
print(classification_report(y_test, y_pred_baseline_test, target_names=LABEL_NAMES))

f1_baseline_test = f1_score(y_test, y_pred_baseline_test, average='macro')
print(f'F1-Macro (teste): {f1_baseline_test:.4f}')

In [ ]:
# ── Comparação lado a lado ────────────────────────────────────────────────────
resultados = []
for nome, y_pred in [('Baseline (TF-IDF + LR)', y_pred_baseline_test), ('BERTimbau', y_pred_bert)]:
    resultados.append({
        'Modelo'      : nome,
        'Acurácia'    : accuracy_score(y_test, y_pred),
        'F1-Macro'    : f1_score(y_test, y_pred, average='macro'),
        'F1-Weighted' : f1_score(y_test, y_pred, average='weighted'),
        'F1-Negativo' : f1_score(y_test, y_pred, average=None)[0],
        'F1-Neutro'   : f1_score(y_test, y_pred, average=None)[1],
        'F1-Positivo' : f1_score(y_test, y_pred, average=None)[2],
    })

df_resultados = pd.DataFrame(resultados).set_index('Modelo')
display(
    df_resultados.style
    .format('{:.4f}')
    .background_gradient(cmap='Oranges', axis=0)
    .set_caption('Comparação de Modelos — Conjunto de Teste')
    .set_table_styles([{
        'selector': 'caption',
        'props': [('font-size', '14px'), ('font-weight', 'bold'), ('color', '#E24307')]
    }])
)

In [ ]:
# ── Matrizes de confusão lado a lado ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Matrizes de Confusão — Conjunto de Teste', fontweight='bold')

for ax, nome, y_pred in [
    (axes[0], 'Baseline (TF-IDF + LR)', y_pred_baseline_test),
    (axes[1], 'BERTimbau',              y_pred_bert)
]:
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=LABEL_NAMES)
    disp.plot(ax=ax, colorbar=False, cmap='Oranges')
    ax.set_title(nome, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ── Análise dos erros mais frequentes — BERTimbau ────────────────────────────
df_test = pd.DataFrame({
    'comentario': X_test,
    'label_real' : y_test,
    'pred_bert'  : y_pred_bert
})
df_test['real_nome'] = df_test['label_real'].map({i: n for i, n in enumerate(LABEL_NAMES)})
df_test['pred_nome'] = df_test['pred_bert'].map({i: n for i, n in enumerate(LABEL_NAMES)})
df_erros = df_test[df_test['label_real'] != df_test['pred_bert']].copy()

print(f'Total de erros do BERTimbau no teste: {len(df_erros):,} ({len(df_erros)/len(df_test)*100:.1f}%)')
print()
print('Erros por par (real → predito):')
pares = df_erros.groupby(['real_nome', 'pred_nome']).size().reset_index(name='qtd').sort_values('qtd', ascending=False)
for _, row in pares.iterrows():
    print(f'  {row["real_nome"]:<10} → {row["pred_nome"]:<10}: {row["qtd"]:,}')

print()
print('Exemplos do erro mais frequente:')
erro_top = pares.iloc[0]
exemplos = df_erros[
    (df_erros['real_nome'] == erro_top['real_nome']) &
    (df_erros['pred_nome'] == erro_top['pred_nome'])
]['comentario'].head(5).tolist()
for ex in exemplos:
    print(f'  > {str(ex)[:180]}')

**Análise dos erros — BERTimbau**

O principal erro do modelo é a confusão entre **Neutro e Negativo**: comentários neutros classificados como negativos. Isso é esperado, pois comentários neutros frequentemente contêm críticas pontuais combinadas com elogios, que o modelo tende a classificar como negativa dado o peso elevado da classe Negativo.

O inverso, negativos classificados como positivos, não ocorre muito, o que indica que o modelo é conservador: quando prediz Positivo, erra pouco. Para o negócio, isso é relevante, pois o risco de "esconder" um cliente insatisfeito atrás de uma predição positiva é baixo.

**Conclusão comparativa:**

O BERTimbau superou o baseline em F1-Macro (**0.5903 vs. 0.5184**, +0.072) e apresentou melhora significativa na classe Negativo (+0.18 de F1), que é a mais crítica para o negócio.



<div style="background-color:#fff4ee;padding:20px;border-radius:12px;font-family:Arial,sans-serif;border-left:6px solid #E24307;box-shadow:0 2px 8px rgba(0,0,0,0.08);">
  <h2 style="color:#E24307;margin:0;font-size:24px;">6. Inferência na Base Completa</h2>
</div>

Com o modelo avaliado, rodamos a inferência sobre **toda** a base de comentários — incluindo os dados que não fizeram parte do treino/validação/teste.

In [ ]:
def inferencia_bert(textos, modelo, tokenizer, max_len, batch_size, device):
    """Roda inferência em lotes e retorna labels preditos e probabilidades."""
    dataset = NpsDataset(textos, [0] * len(textos), tokenizer, max_len)  # labels dummy
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)

    modelo.eval()
    all_preds, all_probs = [], []

    with torch.no_grad():
        for i, batch in enumerate(loader):
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            out  = modelo(input_ids=ids, attention_mask=mask)
            probs = torch.softmax(out.logits, dim=1).cpu().numpy()
            preds = probs.argmax(axis=1)
            all_preds.extend(preds)
            all_probs.extend(probs)
            if (i + 1) % 100 == 0:
                print(f'  Batch {i+1}/{len(loader)}...')

    return np.array(all_preds), np.array(all_probs)


print(f'Rodando inferência em {len(df_com):,} comentários...')
pred_labels, pred_probs = inferencia_bert(
    df_com['comentario_limpo'].tolist(),
    model, tokenizer, MAX_LEN, BATCH_SIZE, device
)

df_com = df_com.copy()
df_com['sentimento_pred']     = pred_labels
df_com['sentimento_pred_nome']= pd.Series(pred_labels).map({i: n for i, n in enumerate(LABEL_NAMES)}).values
df_com['prob_negativo']       = pred_probs[:, 0]
df_com['prob_neutro']         = pred_probs[:, 1]
df_com['prob_positivo']       = pred_probs[:, 2]

print(f'\n✅ Inferência concluída.')
print('\nDistribuição do sentimento predito:')
dist_pred = df_com['sentimento_pred_nome'].value_counts()
for nome, cnt in dist_pred.items():
    print(f'  {nome:<10}: {cnt:>8,}  ({cnt/len(df_com)*100:.1f}%)')

In [ ]:
# Salva base com predições para uso nas análises de negócio
OUTPUT_PATH = 'df_com_sentimento.csv'
cols_export = [
    'mes_ano', 'data_avaliacao', 'centro_nv2', 'classificacao', 'label',
    'comentario', 'comentario_limpo', 'qtd_palavras',
    'sentimento_pred', 'sentimento_pred_nome',
    'prob_negativo', 'prob_neutro', 'prob_positivo',
    'regiao', 'municipio', 'uf', 'flag'
]
df_com[cols_export].to_csv(OUTPUT_PATH, sep=';', index=False, encoding='utf-8-sig')
print(f'✅ Base exportada: {OUTPUT_PATH}')
print(f'   Shape: {df_com[cols_export].shape}')

<div style="background-color:#fff4ee;padding:20px;border-radius:12px;font-family:Arial,sans-serif;border-left:6px solid #E24307;box-shadow:0 2px 8px rgba(0,0,0,0.08);">
  <h2 style="color:#E24307;margin:0;font-size:24px;">Resumo Executivo — Modelo de Sentimento</h2>
</div>

In [ ]:
resumo = pd.DataFrame([
    {'Item': 'Granularidade',              'Decisão': 'Ternária: Positivo / Neutro / Negativo'},
    {'Item': 'Ground truth',               'Decisão': 'Classificação NPS (promotor/neutro/detrator)'},
    {'Item': 'Entrada do modelo',          'Decisão': 'Texto original (com acentos e pontuação)'},
    {'Item': 'Split',                      'Decisão': '70% treino | 15% validação | 15% teste (estratificado)'},
    {'Item': 'Desbalanceamento',           'Decisão': 'Pesos manuais na CrossEntropyLoss: [Neg=3.0, Neu=3.5, Pos=1.0]'},
    {'Item': 'Baseline',                   'Decisão': 'TF-IDF (1-2 gramas, 50k features) + LR multinomial'},
    {'Item': 'Modelo final',               'Decisão': 'BERTimbau fine-tuning (max_len=128, epochs=4, lr=3e-5)'},
    {'Item': 'Critério de parada',         'Decisão': 'Menor val_loss entre as épocas'},
    {'Item': 'Métrica principal',          'Decisão': 'F1-Macro (robusto ao desbalanceamento)'},
    {'Item': 'F1-Macro Baseline (teste)',  'Decisão': '0.5184'},
    {'Item': 'F1-Macro BERTimbau (teste)', 'Decisão': '0.5903  (+0.072 vs. baseline)'},
    {'Item': 'F1-Negativo (BERT)',         'Decisão': '0.58  (recall: 0.77 — captura 77% dos clientes insatisfeitos)'},
    {'Item': 'F1-Positivo (BERT)',         'Decisão': '0.90  (classe majoritária bem aprendida)'},
])

display(
    resumo.style
    .hide(axis='index')
    .set_properties(**{'text-align': 'left', 'font-size': '12px'})
    .set_caption('Resumo das Decisões — Modelo de Análise de Sentimento')
    .set_table_styles([{'selector': 'caption', 'props': [('font-size', '14px'), ('font-weight', 'bold'), ('color', '#E24307')]}])
)